# 🎮 Práctica 08: 3D Scatter Plot con Sprites de Pokémons

---

| Campo | Detalle |
|-------|--------|
| **Estudiante** | Francisco Garcia Garcia |
| **Matrícula** | 230758 |
| **Grupo** | 9°A - IDGS |
| **Materia** | Extracción de Conocimiento en Bases de Datos (ECBD) |
| **Fecha** | 06 de Agosto de 2026 |

---

## Objetivo

Construir un **Scatter Plot 3D interactivo** utilizando **Plotly** que permita visualizar y explorar las estadísticas base de los Pokémon (generaciones 1 a 9), diferenciando visualmente cada tipo principal mediante colores, integrando **sprites oficiales** en la información emergente, y aplicando filtros interactivos por generación, tipo y rango de estadísticas para identificar patrones, agrupaciones y valores atípicos.

## Fuentes de Datos

- **Dataset:** [lgreski/pokemonData](https://github.com/lgreski/pokemonData) — 1,215 Pokémon con estadísticas base (Gen 1–9), cortesía de pokemondb.net
- **Sprites:** [PokeAPI Sprites](https://github.com/PokeAPI/sprites) — Imágenes oficiales de cada Pokémon
- **Referencia:** [Unsupervised Learning: K-Means EDA (Kaggle)](https://www.kaggle.com/code/tanmay111999/unsupervised-learning-3-6-clusters-k-means-eda) — Técnicas de visualización 3D con Plotly

---

## 1. Importación de Librerías

Importamos las librerías necesarias para la manipulación, análisis y visualización de datos:
- **Pandas:** Manipulación y análisis de DataFrames
- **NumPy:** Operaciones numéricas y cálculos estadísticos
- **Plotly:** Visualizaciones 3D interactivas (graph_objects y express)
- **IPython.display:** Visualización enriquecida en Jupyter

In [1]:
# ============================================================
# Importación de Librerías
# ============================================================
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from IPython.display import display, HTML, Image
import warnings

# Configuración general
warnings.filterwarnings('ignore')
pio.templates.default = 'plotly_dark'
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

print('✅ Librerías importadas correctamente')
print(f'   📦 Pandas:  {pd.__version__}')
print(f'   📦 NumPy:   {np.__version__}')
print(f'   📦 Plotly:  {pio.__version__ if hasattr(pio, "__version__") else "disponible"}')

✅ Librerías importadas correctamente
   📦 Pandas:  2.2.2
   📦 NumPy:   2.3.1
   📦 Plotly:  disponible


---

## 2. Carga del Dataset

El dataset proviene del repositorio [lgreski/pokemonData](https://github.com/lgreski/pokemonData) en GitHub, que contiene estadísticas básicas de **1,025 Pokémon únicos** (con formas alternativas el total supera los 1,200 registros) de las **generaciones 1 a 9**, recopiladas de [pokemondb.net](https://pokemondb.net).

**Columnas del dataset:**
- `ID` — Número del Pokédex Nacional
- `Name` — Nombre del Pokémon
- `Form` — Forma/variante (Mega, Alolan, Galarian, etc.)
- `Type1`, `Type2` — Tipos principal y secundario
- `Total` — Suma total de estadísticas base
- `HP`, `Attack`, `Defense`, `Sp. Atk`, `Sp. Def`, `Speed` — Estadísticas individuales
- `Generation` — Generación a la que pertenece

In [2]:
# ============================================================
# Carga del Dataset desde el repositorio de GitHub
# ============================================================
url_dataset = 'https://raw.githubusercontent.com/lgreski/pokemonData/master/Pokemon.csv'

# También se puede cargar localmente:
# df = pd.read_csv('Pokemon.csv')

df = pd.read_csv(url_dataset)

print(f'✅ Dataset cargado exitosamente')
print(f'   📊 Registros: {df.shape[0]}')
print(f'   📋 Columnas:  {df.shape[1]}')
print(f'   📁 Fuente:    lgreski/pokemonData (GitHub)')

✅ Dataset cargado exitosamente
   📊 Registros: 1215
   📋 Columnas:  13
   📁 Fuente:    lgreski/pokemonData (GitHub)


---

## 3. Inspección Inicial del Dataset

Realizamos una inspección completa del dataset utilizando las funciones estándar de Pandas para comprender su estructura, tipos de datos y distribución estadística.

### 3.1 Primeros registros (`head()`)

In [3]:
# ============================================================
# Primeros 10 registros del dataset
# ============================================================
df.head(10)

,ID,Name,Form,Type1,Type2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation
0,1,Bulbasaur,,Grass,Poison,318,45,49,49,65,65,45,1
1,2,Ivysaur,,Grass,Poison,405,60,62,63,80,80,60,1
2,3,Venusaur,,Grass,Poison,525,80,82,83,100,100,80,1
3,4,Charmander,,Fire,,309,39,52,43,60,50,65,1
4,5,Charmeleon,,Fire,,405,58,64,58,80,65,80,1
5,6,Charizard,,Fire,Flying,534,78,84,78,109,85,100,1
6,7,Squirtle,,Water,,314,44,48,65,50,64,43,1
7,8,Wartortle,,Water,,405,59,63,80,65,80,58,1
8,9,Blastoise,,Water,,530,79,83,100,85,105,78,1
9,10,Caterpie,,Bug,,195,45,30,35,20,20,45,1


### 3.2 Dimensiones del Dataset (`shape`)

In [4]:
# ============================================================
# Dimensiones del dataset
# ============================================================
print(f'📐 Dimensiones del dataset:')
print(f'   Filas (registros):  {df.shape[0]}')
print(f'   Columnas (campos):  {df.shape[1]}')
print(f'\n📋 Nombres de columnas:')
for i, col in enumerate(df.columns, 1):
    print(f'   {i:2d}. {col}')

📐 Dimensiones del dataset:
   Filas (registros):  1215
   Columnas (campos):  13

📋 Nombres de columnas:
    1. ID
    2. Name
    3. Form
    4. Type1
    5. Type2
    6. Total
    7. HP
    8. Attack
    9. Defense
   10. Sp. Atk
   11. Sp. Def
   12. Speed
   13. Generation


### 3.3 Información del Dataset (`info()`)

In [5]:
# ============================================================
# Información detallada del dataset
# ============================================================
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1215 entries, 0 to 1214
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ID          1215 non-null   int64 
 1   Name        1215 non-null   object
 2   Form        1215 non-null   object
 3   Type1       1215 non-null   object
 4   Type2       1215 non-null   object
 5   Total       1215 non-null   int64 
 6   HP          1215 non-null   int64 
 7   Attack      1215 non-null   int64 
 8   Defense     1215 non-null   int64 
 9   Sp. Atk     1215 non-null   int64 
 10  Sp. Def     1215 non-null   int64 
 11  Speed       1215 non-null   int64 
 12  Generation  1215 non-null   int64 
dtypes: int64(9), object(4)
memory usage: 123.5+ KB


### 3.4 Estadísticas Descriptivas (`describe()`)

In [6]:
# ============================================================
# Estadísticas descriptivas de columnas numéricas
# ============================================================
df.describe().round(2)

,ID,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation
count,1215.00,1215.00,1215.00,1215.00,1215.00,1215.00,1215.00,1215.00,1215.00
mean,501.74,443.10,71.24,81.15,75.01,73.22,72.44,70.03,5.06
std,298.98,121.19,26.93,32.04,30.74,32.76,27.58,30.16,2.60
min,1.00,175.00,1.00,5.00,5.00,10.00,20.00,5.00,1.00
25%,240.50,332.00,52.00,57.00,52.00,50.00,51.00,45.00,3.00
50%,495.00,465.00,70.00,80.00,70.00,65.00,70.00,68.00,5.00
75%,753.50,521.00,85.00,100.00,91.00,95.00,90.00,91.00,7.00
max,1025.00,1125.00,255.00,190.00,250.00,194.00,250.00,200.00,9.00


In [7]:
# ============================================================
# Estadísticas descriptivas de columnas categóricas
# ============================================================
df.describe(include='object')

,Name,Form,Type1,Type2
count,1215,1215,1215,1215
unique,1026,207,18,19
top,Rotom,,Water,
freq,6,985,150,546


---

## 4. Limpieza y Normalización de Datos

Procedemos a limpiar y normalizar el dataset para garantizar la calidad de los datos antes del análisis:
1. Renombrar columnas a formato `snake_case` para consistencia
2. Limpiar espacios en blanco en valores categóricos
3. Identificar y tratar valores nulos
4. Detectar y eliminar registros duplicados

### 4.1 Estado ANTES de la Limpieza

In [8]:
# ============================================================
# Estado del dataset ANTES de la limpieza
# ============================================================
print('=' * 60)
print('📋 ESTADO ANTES DE LA LIMPIEZA')
print('=' * 60)

# Valores nulos
print('\n🔍 Valores nulos por columna:')
nulos_antes = df.isnull().sum()
print(nulos_antes[nulos_antes > 0] if nulos_antes.sum() > 0 else '   No se encontraron valores nulos explícitos')

# Valores en blanco o espacios en Type2
print(f'\n🔍 Valores en blanco/espacios en Type1: {(df["Type1"].str.strip() == "").sum()}')
print(f'🔍 Valores en blanco/espacios en Type2: {(df["Type2"].str.strip() == "").sum()}')

# Duplicados
print(f'\n🔍 Registros duplicados: {df.duplicated().sum()}')

# Columnas actuales
print(f'\n🔍 Nombres de columnas originales: {list(df.columns)}')

# Tipos únicos
print(f'\n🔍 Tipos únicos en Type1: {sorted(df["Type1"].str.strip().unique())}')

📋 ESTADO ANTES DE LA LIMPIEZA

🔍 Valores nulos por columna:
   No se encontraron valores nulos explícitos

🔍 Valores en blanco/espacios en Type1: 0
🔍 Valores en blanco/espacios en Type2: 546

🔍 Registros duplicados: 0

🔍 Nombres de columnas originales: ['ID', 'Name', 'Form', 'Type1', 'Type2', 'Total', 'HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Generation']

🔍 Tipos únicos en Type1: ['Bug', 'Dark', 'Dragon', 'Electric', 'Fairy', 'Fighting', 'Fire', 'Flying', 'Ghost', 'Grass', 'Ground', 'Ice', 'Normal', 'Poison', 'Psychic', 'Rock', 'Steel', 'Water']


### 4.2 Proceso de Limpieza y Normalización

In [9]:
# ============================================================
# Proceso de limpieza y normalización
# ============================================================

# 1. Renombrar columnas a snake_case
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('.', '', regex=False)
)
print('✅ 1. Columnas renombradas a snake_case')
print(f'   {list(df.columns)}')

# 2. Limpiar espacios en blanco en columnas de texto
for col in ['name', 'form', 'type1', 'type2']:
    df[col] = df[col].str.strip()
print('\n✅ 2. Espacios en blanco eliminados de columnas de texto')

# 3. Reemplazar cadenas vacías en type2 por 'None' (Pokémon de un solo tipo)
df['type2'] = df['type2'].replace('', 'None')
print(f'\n✅ 3. Valores vacíos en type2 reemplazados por "None"')
print(f'   Pokémon con un solo tipo: {(df["type2"] == "None").sum()}')
print(f'   Pokémon con dos tipos:    {(df["type2"] != "None").sum()}')

# 4. Reemplazar cadenas vacías en form por 'Standard'
df['form'] = df['form'].replace('', 'Standard')
print(f'\n✅ 4. Valores vacíos en form reemplazados por "Standard"')

# 5. Verificar y eliminar duplicados
duplicados = df.duplicated().sum()
if duplicados > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'\n✅ 5. Se eliminaron {duplicados} registros duplicados')
else:
    print(f'\n✅ 5. No se encontraron registros duplicados')

# 6. Verificar tipos de datos numéricos
cols_numericas = ['id', 'total', 'hp', 'attack', 'defense', 'sp_atk', 'sp_def', 'speed', 'generation']
for col in cols_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')
print(f'\n✅ 6. Tipos de datos numéricos verificados')

✅ 1. Columnas renombradas a snake_case
   ['id', 'name', 'form', 'type1', 'type2', 'total', 'hp', 'attack', 'defense', 'sp_atk', 'sp_def', 'speed', 'generation']

✅ 2. Espacios en blanco eliminados de columnas de texto

✅ 3. Valores vacíos en type2 reemplazados por "None"
   Pokémon con un solo tipo: 546
   Pokémon con dos tipos:    669

✅ 4. Valores vacíos en form reemplazados por "Standard"

✅ 5. No se encontraron registros duplicados

✅ 6. Tipos de datos numéricos verificados


### 4.3 Estado DESPUÉS de la Limpieza

In [10]:
# ============================================================
# Estado del dataset DESPUÉS de la limpieza
# ============================================================
print('=' * 60)
print('📋 ESTADO DESPUÉS DE LA LIMPIEZA')
print('=' * 60)

# Dimensiones
print(f'\n📐 Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')

# Valores nulos
nulos_despues = df.isnull().sum()
print(f'\n🔍 Valores nulos: {nulos_despues.sum()}')

# Duplicados
print(f'🔍 Registros duplicados: {df.duplicated().sum()}')

# Columnas normalizadas
print(f'\n📋 Columnas normalizadas: {list(df.columns)}')

# Tipos de datos
print(f'\n📊 Tipos de datos:')
for col in df.columns:
    print(f'   {col:20s} → {df[col].dtype}')

# Resumen de tipos de Pokémon
print(f'\n🎯 Tipos únicos de Pokémon (Type1): {df["type1"].nunique()}')
print(f'🎯 Generaciones: {sorted(df["generation"].unique())}')

# Muestra final
print(f'\n📄 Muestra del dataset limpio:')
df.head()

📋 ESTADO DESPUÉS DE LA LIMPIEZA

📐 Dimensiones: 1215 filas × 13 columnas

🔍 Valores nulos: 0
🔍 Registros duplicados: 0

📋 Columnas normalizadas: ['id', 'name', 'form', 'type1', 'type2', 'total', 'hp', 'attack', 'defense', 'sp_atk', 'sp_def', 'speed', 'generation']

📊 Tipos de datos:
   id                   → int64
   name                 → object
   form                 → object
   type1                → object
   type2                → object
   total                → int64
   hp                   → int64
   attack               → int64
   defense              → int64
   sp_atk               → int64
   sp_def               → int64
   speed                → int64
   generation           → int64

🎯 Tipos únicos de Pokémon (Type1): 18
🎯 Generaciones: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]

📄 Muestra del dataset limpio:


,id,name,form,type1,type2,total,hp,attack,defense,sp_atk,sp_def,speed,generation
0,1,Bulbasaur,Standard,Grass,Poison,318,45,49,49,65,65,45,1
1,2,Ivysaur,Standard,Grass,Poison,405,60,62,63,80,80,60,1
2,3,Venusaur,Standard,Grass,Poison,525,80,82,83,100,100,80,1
3,4,Charmander,Standard,Fire,None,309,39,52,43,60,50,65,1
4,5,Charmeleon,Standard,Fire,None,405,58,64,58,80,65,80,1


### 4.4 Resumen Visual del Dataset Limpio

In [11]:
# ============================================================
# Distribución de Pokémon por Tipo Principal
# ============================================================
print('📊 Distribución de Pokémon por Tipo Principal:')
print('=' * 50)
tipo_counts = df['type1'].value_counts()
for tipo, count in tipo_counts.items():
    barra = '█' * (count // 5)
    print(f'   {tipo:12s} │ {barra} {count}')

print(f'\n📊 Distribución de Pokémon por Generación:')
print('=' * 50)
gen_counts = df['generation'].value_counts().sort_index()
for gen, count in gen_counts.items():
    barra = '█' * (count // 5)
    print(f'   Gen {int(gen):1d}      │ {barra} {count}')

📊 Distribución de Pokémon por Tipo Principal:
   Water        │ ██████████████████████████████ 150
   Normal       │ ██████████████████████████ 134
   Grass        │ ██████████████████████ 113
   Bug          │ ██████████████████ 91
   Psychic      │ ████████████████ 82
   Fire         │ ███████████████ 76
   Electric     │ ██████████████ 74
   Rock         │ █████████████ 68
   Dark         │ ███████████ 56
   Fighting     │ ██████████ 50
   Poison       │ █████████ 49
   Dragon       │ █████████ 49
   Ghost        │ █████████ 47
   Ground       │ █████████ 47
   Steel        │ █████████ 45
   Ice          │ ████████ 43
   Fairy        │ ██████ 31
   Flying       │ ██ 10

📊 Distribución de Pokémon por Generación:
   Gen 1      │ ██████████████████████████████ 151
   Gen 2      │ ████████████████████ 100
   Gen 3      │ ████████████████████████████ 141
   Gen 4      │ ███████████████████████ 118
   Gen 5      │ █████████████████████████████████ 165
   Gen 6      │ █████████████████████